In [ ]:
#!pip install tensorflow

<h1 align="center"> Detektor Kecurangan Kartu Kredit (Credit Fraud Detector) </h1>



## Pendahuluan
Dalam program ini, kita akan menggunakan berbagai model prediktif untuk melihat seberapa akurat model-model tersebut dalam mendeteksi apakah sebuah transaksi merupakan **pembayaran normal** atau **penipuan**.  
Seperti yang dijelaskan dalam dataset, fitur-fitur telah diskalakan dan nama fiturnya tidak ditampilkan karena alasan privasi.  
Meskipun demikian, kita masih dapat menganalisis beberapa aspek penting dari dataset tersebut. Mari kita mulai!

## Tujuan Kita:
- Memahami distribusi kecil dari data terbatas yang diberikan kepada kita.  
- Membuat **sub-dataframe** dengan rasio 50/50 antara transaksi *Fraud* dan *Non-Fraud* (menggunakan Algoritma *NearMiss*).  
- Menentukan **classifier** yang akan digunakan dan memilih mana yang memiliki akurasi tertinggi.  
- Membangun **Jaringan Saraf Tiruan (Neural Network)** dan membandingkan akurasinya dengan classifier terbaik.  
- Memahami kesalahan umum yang sering terjadi pada dataset yang tidak seimbang (*imbalanced dataset*).  

## Garis Besar (Outline):
I. **Memahami Data Kita**  
   a) [Mengenal Data Kita](#gather)  

II. **Pra-pemrosesan (Preprocessing)**  
   a) [Skala dan Distribusi Data](#distributing)  
   b) [Membagi Data (Train/Test Split)](#splitting)  

III. **Random UnderSampling dan OverSampling**  
   a) [Distribusi dan Korelasi](#correlating)  
   b) [Deteksi Anomali (Anomaly Detection)](#anomaly)  
   c) [Reduksi Dimensi dan Klasterisasi (t-SNE)](#clustering)  
   d) [Classifier](#classifiers)  
   e) [Pendalaman Logistic Regression](#logistic)  
   f) [Oversampling dengan SMOTE](#smote)  

IV. **Pengujian (Testing)**  
   a) [Pengujian dengan Logistic Regression](#testing_logistic)  
   b) [Pengujian Neural Network (Undersampling vs Oversampling)](#neural_networks)

## Koreksi terhadap Kesalahan Umum pada *Imbalanced Datasets*:
- Jangan pernah menguji model pada dataset yang sudah di-*oversample* atau *undersample*.  
- Jika menggunakan **cross-validation**, lakukan proses oversampling atau undersampling **selama** proses cross-validation, bukan sebelumnya!  
- Jangan menggunakan **accuracy score** sebagai metrik utama untuk dataset yang tidak seimbang, karena hasilnya bisa tinggi namun menyesatkan.  
  Gunakan metrik seperti **f1-score**, **precision/recall**, atau **confusion matrix** sebagai gantinya.

## Referensi:
- *Hands-On Machine Learning with Scikit-Learn & TensorFlow* oleh Aurélien Géron (O'Reilly, 2017).  
- [**Machine Learning - Over & Undersampling - Python/Scikit/Scikit-Imblearn**](https://www.youtube.com/watch?v=DQC_YE3I5ig&t=794s) oleh *Coding-Maniac*.  
- [**AUPRC, 5-Fold CV, and Resampling Metho**]()


## Memahami Data Kita (Gather Sense of Our Data)
<a id="gather"></a>

Hal pertama yang harus kita lakukan adalah memperoleh **gambaran dasar** dari data kita.  
Perlu diingat bahwa, kecuali untuk kolom **transaction** dan **amount**, kita tidak mengetahui makna dari kolom-kolom lainnya (karena alasan privasi).  
Satu-satunya hal yang kita tahu adalah bahwa kolom-kolom tersebut sudah melalui proses **scaling**.

### Ringkasan:
- Jumlah transaksi relatif **kecil**. Rata-rata nilai transaksi adalah sekitar **USD 88**.  
- Tidak terdapat nilai **Null**, sehingga kita tidak perlu melakukan penggantian nilai yang hilang.  
- Sebagian besar transaksi merupakan **Non-Fraud** (sekitar **99.83%**), sedangkan transaksi **Fraud** hanya terjadi sekitar **0.17%** dari total data.

### Aspek Teknis Fitur:
- **Transformasi PCA:** Berdasarkan deskripsi dataset, semua fitur (kecuali *time* dan *amount*) telah melalui proses transformasi **PCA (Principal Component Analysis)** — yaitu teknik reduksi dimensi.  
- **Scaling:** Perlu diingat bahwa agar PCA dapat diterapkan, fitur-fitur harus terlebih dahulu **diskalakan**. Dalam kasus ini, seluruh fitur `V` sudah melalui proses *scaling* — atau setidaknya diasumsikan demikian oleh pengembang dataset.


In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA, TruncatedSVD
import matplotlib.patches as mpatches
import time
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from imblearn.pipeline import make_pipeline as imbalanced_make_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from imblearn.metrics import classification_report_imbalanced
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score, classification_report, confusion_matrix
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# Load Data
df = pd.read_csv('creditcard.csv')
df.head()

In [ ]:
# Deskripsi data
print(df.describe())
print("\n")
print(df.info())

In [ ]:
# Cek nilai null
print(df.isnull().sum().max())

In [ ]:
# Cek distribusi kelas (Class imbalance)
print(df['Class'].value_counts())
print('\n')
print('No Frauds', round(df['Class'].value_counts()[0]/len(df) * 100, 2), '% of the dataset')
print('Frauds', round(df['Class'].value_counts()[1]/len(df) * 100, 2), '% of the dataset')

In [ ]:
# Visualisasi distribusi kelas
colors = ["#0101DF", "#DF0101"]
sns.countplot(x='Class', data=df)
plt.title('Class Distributions \n (0: No Fraud || 1: Fraud)', fontsize=14)
plt.show()

In [ ]:
# Distribusi Time dan Amount
fig, ax = plt.subplots(1, 2, figsize=(18,4))

amount_val = df['Amount'].values
time_val = df['Time'].values

sns.histplot(amount_val, ax=ax[0], color='r', kde=True)
ax[0].set_title('Distribution of Transaction Amount', fontsize=14)
ax[0].set_xlim([min(amount_val), max(amount_val)])

sns.histplot(time_val, ax=ax[1], color='b', kde=True)
ax[1].set_title('Distribution of Transaction Time', fontsize=14)
ax[1].set_xlim([min(time_val), max(time_val)])

plt.show()

In [ ]:
# Scaling Amount dan Time
std_scaler = StandardScaler()
rob_scaler = RobustScaler()

df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1,1))

df.drop(['Time','Amount'], axis=1, inplace=True)

scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']

df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

df.head()

In [ ]:
# Split data dengan stratified sampling
X = df.drop('Class', axis=1)
y = df['Class']

sss = StratifiedKFold(n_splits=5, random_state=None, shuffle=False)

for train_index, test_index in sss.split(X, y):
    print("Train:", train_index, "Test:", test_index)
    original_Xtrain, original_Xtest = X.iloc[train_index], X.iloc[test_index]
    original_ytrain, original_ytest = y.iloc[train_index], y.iloc[test_index]

# Konversi ke array
original_Xtrain = original_Xtrain.values
original_Xtest = original_Xtest.values
original_ytrain = original_ytrain.values
original_ytest = original_ytest.values

# Cek distribusi label
train_unique_label, train_counts_label = np.unique(original_ytrain, return_counts=True)
test_unique_label, test_counts_label = np.unique(original_ytest, return_counts=True)

print('-' * 100)

print('Label Distributions: \n')
print(train_counts_label/ len(original_ytrain))
print(test_counts_label/ len(original_ytest))

In [ ]:
# Random Under-Sampling
df = df.sample(frac=1)

fraud_df = df.loc[df['Class'] == 1]
non_fraud_df = df.loc[df['Class'] == 0][:492]

normal_distributed_df = pd.concat([fraud_df, non_fraud_df])

new_df = normal_distributed_df.sample(frac=1, random_state=42)

new_df.head()

In [ ]:
# Cek distribusi kelas setelah undersampling
print('Distribusi kelas setelah undersampling:')
print(new_df['Class'].value_counts()/len(new_df))

sns.countplot(x='Class', data=new_df)
plt.title('Equally Distributed Classes', fontsize=14)
plt.show()

In [ ]:
# Correlation Matrix
f, (ax1, ax2) = plt.subplots(2, 1, figsize=(24,20))

corr = df.corr()
sns.heatmap(corr, cmap='coolwarm_r', annot_kws={'size':20}, ax=ax1)
ax1.set_title("Imbalanced Correlation Matrix \n (don't use for reference)", fontsize=14)

sub_sample_corr = new_df.corr()
sns.heatmap(sub_sample_corr, cmap='coolwarm_r', annot_kws={'size':20}, ax=ax2)
ax2.set_title('SubSample Correlation Matrix \n (use for reference)', fontsize=14)
plt.show()

In [ ]:
# Negative Correlations
f, axes = plt.subplots(ncols=4, figsize=(20,4))

sns.boxplot(x="Class", y="V17", data=new_df, ax=axes[0])
axes[0].set_title('V17 vs Class Negative Correlation')

sns.boxplot(x="Class", y="V14", data=new_df, ax=axes[1])
axes[1].set_title('V14 vs Class Negative Correlation')

sns.boxplot(x="Class", y="V12", data=new_df, ax=axes[2])
axes[2].set_title('V12 vs Class Negative Correlation')

sns.boxplot(x="Class", y="V10", data=new_df, ax=axes[3])
axes[3].set_title('V10 vs Class Negative Correlation')

plt.show()

In [ ]:
# Positive correlations
f, axes = plt.subplots(ncols=4, figsize=(20,4))

sns.boxplot(x="Class", y="V11", data=new_df, ax=axes[0])
axes[0].set_title('V11 vs Class Positive Correlation')

sns.boxplot(x="Class", y="V4", data=new_df, ax=axes[1])
axes[1].set_title('V4 vs Class Positive Correlation')

sns.boxplot(x="Class", y="V2", data=new_df, ax=axes[2])
axes[2].set_title('V2 vs Class Positive Correlation')

sns.boxplot(x="Class", y="V19", data=new_df, ax=axes[3])
axes[3].set_title('V19 vs Class Positive Correlation')

plt.show()

In [ ]:
# Anomaly Detection (Remove outliers from V14)
v14_fraud = new_df['V14'].loc[new_df['Class'] == 1].values
q25, q75 = np.percentile(v14_fraud, 25), np.percentile(v14_fraud, 75)
print('Percentile 25: {}'.format(q25))
print('Percentile 75: {}'.format(q75))
v14_iqr = q75 - q25
print('IQR: {}'.format(v14_iqr))

v14_cut_off = v14_iqr * 1.5
lower, upper = q25 - v14_cut_off, q75 + v14_cut_off
print('Cut Off: {}'.format(v14_cut_off))
print('V14 Lower: {}'.format(lower))
print('V14 Upper: {}'.format(upper))

outliers = [x for x in v14_fraud if x < lower or x > upper]
print('Feature V14 Outliers for Fraud Cases: {}'.format(len(outliers)))

new_df = new_df.drop(new_df[(new_df['V14'] > upper) | (new_df['V14'] < lower)].index)
print('Number of instances after outliers removal: {}'.format(len(new_df)))

In [ ]:
# Dimensionality Reduction (t-SNE, PCA, TruncatedSVD)
X = new_df.drop('Class', axis=1)
y = new_df['Class']

# t-SNE
t0 = time.time()
X_reduced_tsne = TSNE(n_components=2, random_state=42).fit_transform(X.values)
t1 = time.time()
print("t-SNE took {:.2} s".format(t1 - t0))

# PCA
t0 = time.time()
X_reduced_pca = PCA(n_components=2, random_state=42).fit_transform(X.values)
t1 = time.time()
print("PCA took {:.2} s".format(t1 - t0))

# TruncatedSVD
t0 = time.time()
X_reduced_svd = TruncatedSVD(n_components=2, algorithm='randomized', random_state=42).fit_transform(X.values)
t1 = time.time()
print("Truncated SVD took {:.2} s".format(t1 - t0))

In [ ]:
# Visualisasi t-SNE, PCA, dan TruncatedSVD
f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24,6))
f.suptitle('Clusters using Dimensionality Reduction', fontsize=14)

blue_patch = mpatches.Patch(color='#0A0AFF', label='No Fraud')
red_patch = mpatches.Patch(color='#AF0000', label='Fraud')

# t-SNE
ax1.scatter(X_reduced_tsne[:,0], X_reduced_tsne[:,1], c=(y == 0), cmap='coolwarm', label='No Fraud', linewidths=2)
ax1.scatter(X_reduced_tsne[:,0], X_reduced_tsne[:,1], c=(y == 1), cmap='coolwarm', label='Fraud', linewidths=2)
ax1.set_title('t-SNE', fontsize=14)
ax1.grid(True)
ax1.legend(handles=[blue_patch, red_patch])

# PCA
ax2.scatter(X_reduced_pca[:,0], X_reduced_pca[:,1], c=(y == 0), cmap='coolwarm', label='No Fraud', linewidths=2)
ax2.scatter(X_reduced_pca[:,0], X_reduced_pca[:,1], c=(y == 1), cmap='coolwarm', label='Fraud', linewidths=2)
ax2.set_title('PCA', fontsize=14)
ax2.grid(True)
ax2.legend(handles=[blue_patch, red_patch])

# TruncatedSVD
ax3.scatter(X_reduced_svd[:,0], X_reduced_svd[:,1], c=(y == 0), cmap='coolwarm', label='No Fraud', linewidths=2)
ax3.scatter(X_reduced_svd[:,0], X_reduced_svd[:,1], c=(y == 1), cmap='coolwarm', label='Fraud', linewidths=2)
ax3.set_title('Truncated SVD', fontsize=14)
ax3.grid(True)
ax3.legend(handles=[blue_patch, red_patch])

plt.show()

In [ ]:
# Classifier Implementation with Cross-Validation
X = new_df.drop('Class', axis=1)
y = new_df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

classifiers = {
    "LogisiticRegression": LogisticRegression(),
    "KNearest": KNeighborsClassifier(),
    "Support Vector Classifier": SVC(),
    "DecisionTreeClassifier": DecisionTreeClassifier()
}

for key, classifier in classifiers.items():
    classifier.fit(X_train, y_train)
    training_score = cross_val_score(classifier, X_train, y_train, cv=5)
    print("Classifiers: ", classifier.__class__.__name__, "Has a training score of", round(training_score.mean(), 2) * 100, "% accuracy score")

In [ ]:
# Hyperparameter Tuning with GridSearchCV
log_reg_params = {"penalty": ['l2'], 'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000], "solver": ["lbfgs", "liblinear"]}
grid_log_reg = GridSearchCV(LogisticRegression(max_iter=1000), log_reg_params)
grid_log_reg.fit(X_train, y_train)
log_reg = grid_log_reg.best_estimator_

knears_params = {"n_neighbors": list(range(2,5,1)), 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}
grid_knears = GridSearchCV(KNeighborsClassifier(), knears_params)
grid_knears.fit(X_train, y_train)
knears_neighbors = grid_knears.best_estimator_

svc_params = {'C': [0.5, 0.7, 0.9, 1], 'kernel': ['rbf', 'poly', 'sigmoid', 'linear']}
grid_svc = GridSearchCV(SVC(), svc_params)
grid_svc.fit(X_train, y_train)
svc = grid_svc.best_estimator_

tree_params = {"criterion": ["gini", "entropy"], "max_depth": list(range(2,4,1)), 
              "min_samples_leaf": list(range(5,7,1))}
grid_tree = GridSearchCV(DecisionTreeClassifier(), tree_params)
grid_tree.fit(X_train, y_train)
tree_clf = grid_tree.best_estimator_

print("Logistic Regression Best Params:", grid_log_reg.best_params_)
print("KNearest Best Params:", grid_knears.best_params_)
print("SVC Best Params:", grid_svc.best_params_)
print("Decision Tree Best Params:", grid_tree.best_params_)

In [ ]:
# Evaluate classifiers with metrics
log_reg_score = cross_val_score(log_reg, X_train, y_train, cv=5)
print('Logistic Regression Cross Validation Score: ', round(log_reg_score.mean() * 100, 2).astype(str) + '%')

knears_score = cross_val_score(knears_neighbors, X_train, y_train, cv=5)
print('Knears Neighbors Cross Validation Score', round(knears_score.mean() * 100, 2).astype(str) + '%')

svc_score = cross_val_score(svc, X_train, y_train, cv=5)
print('Support Vector Classifier Cross Validation Score', round(svc_score.mean() * 100, 2).astype(str) + '%')

tree_score = cross_val_score(tree_clf, X_train, y_train, cv=5)
print('DecisionTree Classifier Cross Validation Score', round(tree_score.mean() * 100, 2).astype(str) + '%')

In [ ]:
# Predictions and confusion matrix
y_pred_log_reg = log_reg.predict(X_test)
y_pred_knear = knears_neighbors.predict(X_test)
y_pred_svc = svc.predict(X_test)
y_pred_tree = tree_clf.predict(X_test)

log_reg_cf = confusion_matrix(y_test, y_pred_log_reg)
kneighbors_cf = confusion_matrix(y_test, y_pred_knear)
svc_cf = confusion_matrix(y_test, y_pred_svc)
tree_cf = confusion_matrix(y_test, y_pred_tree)

fig, ax = plt.subplots(2, 2,figsize=(22,12))

sns.heatmap(log_reg_cf, ax=ax[0][0], annot=True, cmap=plt.cm.copper)
ax[0, 0].set_title("Logistic Regression \n Confusion Matrix", fontsize=14)
ax[0, 0].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[0, 0].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(kneighbors_cf, ax=ax[0][1], annot=True, cmap=plt.cm.copper)
ax[0][1].set_title("KNearest Neighbors \n Confusion Matrix", fontsize=14)
ax[0][1].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[0][1].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(svc_cf, ax=ax[1][0], annot=True, cmap=plt.cm.copper)
ax[1][0].set_title("Support Vector Classifier \n Confusion Matrix", fontsize=14)
ax[1][0].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[1][0].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(tree_cf, ax=ax[1][1], annot=True, cmap=plt.cm.copper)
ax[1][1].set_title("Decision Tree Classifier \n Confusion Matrix", fontsize=14)
ax[1][1].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[1][1].set_yticklabels(['', ''], fontsize=14, rotation=360)

plt.show()

In [ ]:
# Classification Report
print('Logistic Regression:')
print(classification_report(y_test, y_pred_log_reg))

print('KNears Neighbors:')
print(classification_report(y_test, y_pred_knear))

print('Support Vector Classifier:')
print(classification_report(y_test, y_pred_svc))

print('Decision Tree:')
print(classification_report(y_test, y_pred_tree))

In [ ]:
# SMOTE (Synthetic Minority Over-sampling Technique)
df_original = pd.read_csv('creditcard.csv')
df_original['scaled_amount'] = rob_scaler.fit_transform(df_original['Amount'].values.reshape(-1,1))
df_original['scaled_time'] = rob_scaler.fit_transform(df_original['Time'].values.reshape(-1,1))
df_original.drop(['Time','Amount'], axis=1, inplace=True)
scaled_amount = df_original['scaled_amount']
scaled_time = df_original['scaled_time']
df_original.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df_original.insert(0, 'scaled_amount', scaled_amount)
df_original.insert(1, 'scaled_time', scaled_time)

X = df_original.drop('Class', axis=1)
y = df_original['Class']

print('Original dataset shape %s' % Counter(y))

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)
print('Resampled dataset shape %s' % Counter(y_res))

In [ ]:
# Neural Network Implementation using Keras
try:
    from keras.models import Sequential
    from keras.layers import Dense, Dropout
    from keras.optimizers import Adam
    
    # Prepare data from SMOTE resampled dataset
    X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(X_res, y_res, test_size=0.2, random_state=42)
    
    # Build Neural Network
    model = Sequential([
        Dense(32, input_dim=X_train_nn.shape[1], activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    
    # Train model
    history = model.fit(X_train_nn, y_train_nn, epochs=20, batch_size=32, validation_split=0.2, verbose=1)
    
    # Evaluate
    y_pred_nn = model.predict(X_test_nn)
    y_pred_nn_binary = (y_pred_nn > 0.5).astype(int)
    
    print('\nNeural Network Classification Report:')
    print(classification_report(y_test_nn, y_pred_nn_binary))
    print('ROC AUC Score:', roc_auc_score(y_test_nn, y_pred_nn))
except ImportError:
    print("Keras/TensorFlow not available. Skipping Neural Network implementation.")

In [ ]:
# Plot training history (if neural network was trained)
try:
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.show()
except:
    print("No training history available (neural network was not trained).")

## Kesimpulan (Conclusion)

### Ringkasan Hasil:
1. **Distribusi Data**: Dataset sangat tidak seimbang dengan 99.83% transaksi normal dan hanya 0.17% transaksi fraud.
2. **Preprocessing**: Kolom Amount dan Time telah diskalakan menggunakan RobustScaler untuk menangani outlier.
3. **Random Undersampling**: Menggunakan teknik undersampling untuk mendapatkan dataset seimbang (50:50).
4. **Anomaly Detection**: Menghapus outlier dari fitur V14 untuk meningkatkan performa model.
5. **Dimensionality Reduction**: t-SNE, PCA, dan TruncatedSVD digunakan untuk visualisasi cluster.
6. **Classifier Comparison**: 
   - Logistic Regression
   - K-Nearest Neighbors
   - Support Vector Classifier
   - Decision Tree Classifier
7. **SMOTE Oversampling**: Teknik oversampling untuk menangani dataset tidak seimbang.
8. **Neural Network**: Menggunakan Keras untuk membangun model deep learning (jika tersedia).

### Metrik Evaluasi:
- Accuracy Score
- Precision, Recall, F1-Score
- Confusion Matrix
- ROC AUC Score

### Rekomendasi:
- Gunakan dataset yang sudah di-undersample atau oversample untuk training, tetapi gunakan dataset asli untuk testing.
- Fokus pada metrik precision dan recall untuk dataset yang tidak seimbang, bukan accuracy.
- Logistic Regression cenderung memberikan hasil yang baik dan stabil untuk kasus ini.